# LLM, Embedder and Scorer

The three model interfaces RAGU is built on. Every RAGU component takes these
interfaces rather than a concrete provider, so swapping a backend never touches
the pipeline. This notebook exercises each one directly, without a knowledge graph.

- **`LLMOpenAI`** — plain text, structured (Pydantic) output, streaming and batched
  generation.
- **`EmbedderOpenAI`** — single and batched embeddings, dimension auto-detection,
  automatic truncation to the model context window.
- **`ScorerOpenAI`** — reranking a candidate list against a query. It posts to
  `{base_url}score`, the endpoint vLLM- and Infinity-style rerank servers expose;
  it is **not** part of the OpenAI API. For a local reranker with no server, use
  `ScorerCrossEncoder` from the same module (needs the `[local]` extra).

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`,
`RERANKER_MODEL_NAME`, and optionally `OPENAI_BASE_URL`.

In [ ]:
import math
import os

from pydantic import BaseModel, Field

from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.models.scorer import ScorerOpenAI

TEXT = (
    "Dennis Ritchie created the C programming language at Bell Labs in the early "
    "1970s and, together with Ken Thompson, developed the Unix operating system."
)

QUERY = "Who created the C programming language?"

CANDIDATES = [
    "C was designed by Dennis Ritchie at Bell Labs.",
    "Python is a high-level interpreted language created by Guido van Rossum.",
    "Unix was developed at Bell Labs by Ken Thompson and Dennis Ritchie.",
    "The Eiffel Tower is located in Paris, France.",
]


def cosine(left: list[float], right: list[float]) -> float:
    dot = sum(a * b for a, b in zip(left, right))
    norm = math.sqrt(sum(a * a for a in left)) * math.sqrt(sum(b * b for b in right))
    return dot / norm if norm else 0.0

## One client, three adapters

Rate limits, retries and caching are enforced once for the whole notebook rather
than per model. Constructor kwargs (like `temperature`) become per-call defaults;
a kwarg passed to the call itself overrides them.

`dim=None` (the default) means "probe the model once during `initialize()`".

In [ ]:
client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)

llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"], temperature=0.0)

embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

scorer = ScorerOpenAI(client=client, model_name=os.environ["RERANKER_MODEL_NAME"])

## LLM — plain text

Conversations are plain OpenAI-format dicts. Anything producing that shape works,
including `ChatMessages.to_openai()` from `ragu.common.prompts`.

In [ ]:
conversation = [
    {"role": "system", "content": "You are a concise technical assistant."},
    {"role": "user", "content": f"Summarize in one sentence:\n{TEXT}"},
]
print(await llm.chat_completion(conversation))

## LLM — structured output

Passing a Pydantic model switches the call to structured output and returns a
validated instance instead of a string.

In [ ]:
class Summary(BaseModel):
    """Structured output schema requested from the LLM."""

    title: str = Field(description="Short title of the text")
    key_points: list[str] = Field(description="Three key points, one sentence each")


summary = await llm.chat_completion(
    [{"role": "user", "content": TEXT}],
    output_schema=Summary,
)

print(f"title: {summary.title}")
for point in summary.key_points:
    print(f"  - {point}")

## LLM — streaming

Text-only by design: a schema cannot be validated before the response is complete.

In [ ]:
async for delta in llm.stream_chat_completion(conversation):
    print(delta, end="", flush=True)

## LLM — batched

Concurrency is bounded by the client's rate limits, not by this call.

In [ ]:
questions = ["Who created C?", "Where was Unix developed?"]
answers = await llm.batch_chat_completion(
    [[{"role": "user", "content": f"Answer in one word: {q}"}] for q in questions],
    desc="LLM batch",
)
for question, answer in zip(questions, answers):
    print(f"{question} -> {answer}")

## Embedder

`dim` is known only after `initialize()` unless it was passed to the constructor —
storage backends need it up front to size their collections.

`batch_embed_text` groups texts into API-level sub-batches (`batch_size`) and caps
concurrent calls (`max_concurrent_batches`), so a few thousand texts cost a handful
of requests instead of a few thousand.

In [ ]:
print(f"dimension: {embedder.dim}")

query_vector = await embedder.embed_text(QUERY)
candidate_vectors = await embedder.batch_embed_text(CANDIDATES, desc="Embedding")

for candidate, vector in zip(CANDIDATES, candidate_vectors):
    print(f"{cosine(query_vector, vector):+.4f}  {candidate}")

## Scorer (reranker)

`score()` returns `(original_index, score)` sorted best-first, so the caller can
reorder its own list without losing provenance. Compare this ordering against the
cosine ordering above — a cross-encoder reads the query and the candidate
*together*, which is why it separates the near-misses that embeddings blur.

In [ ]:
for index, score in await scorer.score(QUERY, CANDIDATES):
    print(f"{score:+.4f}  {CANDIDATES[index]}")